In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import TensorDataset, DataLoader
import numpy as np
from pathlib import Path

In [3]:
# Load saved dataset
data=np.load('data/digit_splits.npz')

X_train=data['X_train']
y_train=data['y_train']

X_val=data['X_val']
y_val=data['y_val']

X_test=data['X_test']
y_test=data['y_test']

print("Training data:", X_train.shape)
print("Validation data:", X_val.shape)
print("Testing data:", X_test.shape)

Training data: (1257, 8, 8)
Validation data: (270, 8, 8)
Testing data: (270, 8, 8)


In [4]:
# Convert training data
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)

# Convert validation data
X_val = torch.tensor(X_val, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.long)

# Convert testing data
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.long)

In [5]:
# Divide each image into 16 patches
patches = X_train.unfold(1, 2, 2).unfold(2, 2, 2)

# Rearrange patches
patches = patches.permute(0, 1, 2, 3, 4)

# Flatten each 2x2 patch into 4 numbers
patches = patches.reshape(-1, 16, 4)

print("Original images:", X_train.shape)
print("Patch shape:", patches.shape)

Original images: torch.Size([1257, 8, 8])
Patch shape: torch.Size([1257, 16, 4])


In [6]:
class PatchEmbedding(nn.Module):

    def __init__(self):
        super().__init__()

        self.projection = nn.Linear(4, 16)

    def forward(self, patches):

        return self.projection(patches)

In [7]:
# Create sample patches
sample_patches = torch.rand(2, 16, 4)

# Create embedding layer
embedding = PatchEmbedding()

# Convert patches into embeddings
embedded_patches = embedding(sample_patches)

print("Original shape:", sample_patches.shape)
print("Embedded shape:", embedded_patches.shape)

Original shape: torch.Size([2, 16, 4])
Embedded shape: torch.Size([2, 16, 16])


In [9]:
class PositionalEmbedding(nn.Module):

    def __init__(self):
        super().__init__()

        self.position = nn.Parameter(
            torch.zeros(1, 16, 16)
        )

    def forward(self, x):

        return x + self.position

In [10]:
# Create positional embedding layer
position_embedding = PositionalEmbedding()

# Add position information to embedded patches
positioned_patches = position_embedding(embedded_patches)

print("Before position:", embedded_patches.shape)
print("After position:", positioned_patches.shape)

Before position: torch.Size([2, 16, 16])
After position: torch.Size([2, 16, 16])


In [11]:
class TransformerBlock(nn.Module):

    def __init__(self):
        super().__init__()

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=16,
            nhead=4,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )

    def forward(self, x):

        return self.transformer(x)

In [12]:
# Create transformer
transformer = TransformerBlock()

# Pass positioned patches through transformer
transformed_patches = transformer(positioned_patches)

print("Before Transformer:", positioned_patches.shape)
print("After Transformer:", transformed_patches.shape)

Before Transformer: torch.Size([2, 16, 16])
After Transformer: torch.Size([2, 16, 16])


In [13]:
class DigitClassifier(nn.Module):

    def __init__(self):
        super().__init__()

        self.fc = nn.Linear(16, 10)

    def forward(self, x):

        x = x.mean(dim=1)

        return self.fc(x)

In [14]:
class DigitViT(nn.Module):

    def __init__(self):
        super().__init__()

        self.patch_embedding = PatchEmbedding()

        self.position_embedding = PositionalEmbedding()

        self.transformer = TransformerBlock()

        self.classifier = DigitClassifier()

    def forward(self, x):

        # Divide image into 2x2 patches
        x = x.unfold(1, 2, 2).unfold(2, 2, 2)

        # Convert patches into 16 patches with 4 values each
        x = x.reshape(x.shape[0], 16, 4)

        # Convert patches into embeddings
        x = self.patch_embedding(x)

        # Add position information
        x = self.position_embedding(x)

        # Apply self-attention
        x = self.transformer(x)

        # Predict digit
        x = self.classifier(x)

        return x

In [15]:
# Create ViT model
model = DigitViT()

# Create a sample image (2 images of size 8x8)
sample_images = torch.rand(2, 8, 8)

# Pass images through the model
outputs = model(sample_images)

print("Input shape:", sample_images.shape)
print("Output shape:", outputs.shape)
print("Predictions:", outputs)

Input shape: torch.Size([2, 8, 8])
Output shape: torch.Size([2, 10])
Predictions: tensor([[ 0.3267,  0.5125, -0.0567,  0.0069,  0.1968,  0.1067, -0.3687,  0.3648,
          0.1196,  0.1846],
        [ 0.2937,  0.6421,  0.0191, -0.0573,  0.2003,  0.0772, -0.2866,  0.3016,
          0.1183,  0.1885]], grad_fn=<AddmmBackward0>)


In [16]:
# Create training dataset
train_dataset = TensorDataset(X_train, y_train)

# Create training DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

# Create validation dataset
val_dataset = TensorDataset(X_val, y_val)

# Create validation DataLoader
val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Training batches: 40
Validation batches: 9


In [17]:
# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

In [18]:
epochs = 30

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for images, labels in train_loader:

        # Clear previous gradients
        optimizer.zero_grad()

        # Make predictions
        outputs = model(images)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Calculate gradients
        loss.backward()

        # Update model weights
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)

    print(
        f"Epoch [{epoch+1}/{epochs}], "
        f"Loss: {average_loss:.4f}"
    )

Epoch [1/30], Loss: 2.3441
Epoch [2/30], Loss: 2.3105
Epoch [3/30], Loss: 2.3042
Epoch [4/30], Loss: 2.2297
Epoch [5/30], Loss: 1.9619
Epoch [6/30], Loss: 1.6722
Epoch [7/30], Loss: 1.4410
Epoch [8/30], Loss: 1.2951
Epoch [9/30], Loss: 1.1107
Epoch [10/30], Loss: 0.9647
Epoch [11/30], Loss: 0.8026
Epoch [12/30], Loss: 0.6556
Epoch [13/30], Loss: 0.5699
Epoch [14/30], Loss: 0.4912
Epoch [15/30], Loss: 0.4367
Epoch [16/30], Loss: 0.3898
Epoch [17/30], Loss: 0.3166
Epoch [18/30], Loss: 0.3083
Epoch [19/30], Loss: 0.2794
Epoch [20/30], Loss: 0.2429
Epoch [21/30], Loss: 0.2082
Epoch [22/30], Loss: 0.1855
Epoch [23/30], Loss: 0.1828
Epoch [24/30], Loss: 0.1582
Epoch [25/30], Loss: 0.1409
Epoch [26/30], Loss: 0.1409
Epoch [27/30], Loss: 0.1259
Epoch [28/30], Loss: 0.1586
Epoch [29/30], Loss: 0.1174
Epoch [30/30], Loss: 0.1373


In [19]:
# Validate the model
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in val_loader:

        outputs = model(images)

        predictions = torch.argmax(outputs, dim=1)

        correct += (predictions == labels).sum().item()

        total += labels.size(0)

accuracy = (correct / total) * 100

print(f"Validation Accuracy: {accuracy:.2f}%")

Validation Accuracy: 94.81%


In [20]:
# Create models folder
Path("models").mkdir(exist_ok=True)

# Save trained ViT model
torch.save(
    model.state_dict(),
    "models/vit_model.pth"
)

print("ViT model saved successfully!")

ViT model saved successfully!


In [21]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix

In [22]:
# Set model to evaluation mode
model.eval()

# Make predictions on test data
with torch.no_grad():

    test_outputs = model(X_test)

    test_predictions = torch.argmax(test_outputs, dim=1)

# Convert tensors to NumPy
y_true = y_test.numpy()
y_pred = test_predictions.numpy()

# Calculate accuracy
accuracy = accuracy_score(y_true, y_pred)

print("ViT Test Accuracy:", accuracy * 100)

ViT Test Accuracy: 94.07407407407408


In [23]:
# Calculate Macro F1-score
macro_f1 = f1_score(
    y_true,
    y_pred,
    average="macro"
)

print("ViT Macro F1-score:", macro_f1)

ViT Macro F1-score: 0.9408437434582988


In [24]:
# Calculate confusion matrix
cm = confusion_matrix(y_true, y_pred)

print("ViT Confusion Matrix:")
print(cm)

ViT Confusion Matrix:
[[27  0  0  0  0  0  0  0  0  0]
 [ 0 26  0  1  0  0  0  0  1  0]
 [ 0  1 23  0  0  0  0  0  2  0]
 [ 0  0  0 27  0  0  0  0  1  0]
 [ 1  0  0  0 24  1  1  0  0  0]
 [ 0  0  0  0  0 27  0  0  0  0]
 [ 1  0  0  0  0  0 26  0  0  0]
 [ 0  0  0  0  2  0  0 25  0  0]
 [ 0  1  0  0  0  0  0  0 24  1]
 [ 0  0  0  0  0  0  0  0  2 25]]


In [25]:
vit_accuracy = accuracy
vit_f1 = macro_f1

In [26]:
import json

with open("models/vit_metrics.json", "w") as f:
    json.dump({
        "vit_accuracy": float(vit_accuracy),
        "vit_f1": float(vit_f1)
    }, f)

print("ViT metrics saved!")

ViT metrics saved!
